In [ ]:
!pip install ujson

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.4/57.4 kB 3.7 MB/s eta 0:00:00


In [ ]:
import torch, gc
gc.collect()
torch.cuda.empty_cache()

In [ ]:
from google.colab import files
uploaded=files.upload()

for fn in uploaded.keys():
  print(f'fichier "{fn}" importe avec succes')
  # dataset_phase1_final_READY.jsonl

In [ ]:
!pip install rouge_score evaluate absl-py


  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.3 MB/s eta 0:00:00
  Created wheel for rouge_score: filename=rouge_score-0.1.2-py3-none-any.whl size=24934 sha256=8d9a24c6e1e3ded836251cbbd58db77ef49094db77ea4290e3d9a400b5491f04
  Stored in directory: /root/.cache/pip/wheels/85/9d/af/01feefbe7d55ef5468796f0c68225b6788e85d9d0a281e7a70
Successfully built rouge_score


In [ ]:
from huggingface_hub import notebook_login
notebook_login()



In [ ]:
from huggingface_hub import snapshot_download

# On télécharge ce qui est sur le Hub vers ton dossier local
snapshot_download(
    repo_id="Fatoumataa/mt5-bambara-resumer-boost1",
    local_dir="./mt5-bambara-resumer-boost1",
    repo_type="model"
)

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

'/content/mt5-bambara-resumer-boost1'

In [ ]:
import os
import random
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    EarlyStoppingCallback
)
from transformers.trainer_utils import get_last_checkpoint

# 1. SEED (Reproductibilité)
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

# 2. CONFIGURATION
source_model_id = "Fatoumataa/mt5-bambara-resumer-final"
repo_id_boost = "Fatoumataa/mt5-bambara-resumer-boost1"
dataset_path = "/content/dataset_phase1_final_READY.jsonl"
output_dir = "./mt5-bambara-resumer-boost1"

# 3. CHARGEMENT DU MODÈLE ET DU TOKENIZER
tokenizer = AutoTokenizer.from_pretrained(source_model_id, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(source_model_id)

# 4. PRÉPARATION DU DATASET
df = pd.read_json(dataset_path, lines=True)
df["id"] = df["id"].astype(str)
dataset = Dataset.from_pandas(df).train_test_split(test_size=0.1, seed=seed)

def preprocess_function(examples):
    model_inputs = tokenizer(examples["input_text"], max_length=512, truncation=True)
    labels = tokenizer(text_target=examples["target_text"], max_length=128, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_dataset = dataset.map(preprocess_function, batched=True, remove_columns=dataset["train"].column_names)

# 5. MÉTRIQUES SÉCURISÉES (Calcul du ROUGE avec protection vocabulaire)
rouge_metric = evaluate.load("rouge")
def compute_metrics(eval_preds):
    preds, labels = eval_preds
    if isinstance(preds, tuple): preds = preds[0]

    vocab_size = tokenizer.vocab_size
    # Protection contre les indices hors vocabulaire
    preds = np.where((preds >= 0) & (preds < vocab_size), preds, tokenizer.pad_token_id)

    decoded_preds = tokenizer.batch_decode(preds, skip_special_tokens=True)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)
    labels = np.where((labels >= 0) & (labels < vocab_size), labels, tokenizer.pad_token_id)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    result = rouge_metric.compute(predictions=decoded_preds, references=decoded_labels)
    return {k: round(v, 4) for k, v in result.items()}

# 6. TRAINING ARGUMENTS (STRATÉGIE BOOST CHIRURGICALE)
training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,           # Garde uniquement les 2 derniers checkpoints (économie disque)
    logging_steps=25,

    # --- HYPERPARAMÈTRES BOOST ---
    learning_rate=2e-5,
    lr_scheduler_type="linear",
    warmup_steps=50,
    weight_decay=0.01,
    label_smoothing_factor=0.05,
    max_grad_norm=1.0,            # Gradient clipping pour la stabilité
    # ------------------------------

    per_device_train_batch_size=2,
    gradient_accumulation_steps=16, # Batch effectif = 32
    per_device_eval_batch_size=2,

    predict_with_generate=True,
    generation_max_length=128,
    generation_num_beams=4,

    num_train_epochs=8,
    fp16=False,                   # Désactivé pour mT5 (souvent plus stable en FP32 sur Colab)
    optim="adamw_torch",

    # --- CONFIGURATION HUB & REPRISE ---
    push_to_hub=True,
    hub_model_id=repo_id_boost,
    hub_strategy="checkpoint",    # Envoie les dossiers checkpoints sur le Hub à chaque epoch
    # ------------------------------------

    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,

    seed=seed,
    data_seed=seed,
    report_to="none"
)
last_checkpoint = os.path.join(output_dir, "last-checkpoint")

# On vérifie si ce dossier spécifique existe et contient l'état de l'entraînement
if os.path.exists(os.path.join(last_checkpoint, "trainer_state.json")):
    print(f"✅ REPRISE FORCÉE DEPUIS : {last_checkpoint}")

    # Nettoyage du scaler pour éviter les bugs de précision
    scaler_path = os.path.join(last_checkpoint, "scaler.pt")
    if os.path.exists(scaler_path):
        os.remove(scaler_path)
        print("🧹 Fichier scaler.pt supprimé.")
else:
    print("🆕 DOSSIER 'last-checkpoint' NON TROUVÉ : Début à partir du modèle source.")
    last_checkpoint = None

# 8. INITIALISATION ET ENTRAÎNEMENT
trainer = Seq2SeqTrainer(

    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"],
    data_collator=DataCollatorForSeq2Seq(tokenizer, model=model),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("🚀 Lancement du BOOST FINAL - Phase 1.5")
# resume_from_checkpoint sera None si aucun dossier n'existe
trainer.train(resume_from_checkpoint=last_checkpoint)

# SAUVEGARDE ET PUSH FINAL
trainer.push_to_hub(commit_message="Phase 1 Boost - Final Stable Version")

config.json:   0%|          | 0.00/803 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/16.0M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.23G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

...

  ...mer-boost1/tokenizer.json:  52%|#####1    | 8.33MB / 16.0MB            

  ...-boost1/model.safetensors:   0%|          | 8.28MB / 2.23GB            

  ...ckpoint/model.safetensors:   0%|          | 8.35MB / 2.23GB            

  ...t-checkpoint/optimizer.pt:   0%|          | 16.8MB / 3.43GB            

CommitInfo(commit_url='https://huggingface.co/Fatoumataa/mt5-bambara-resumer-boost1/commit/1bb7b93eb35950babe031720987a13517b5c5572', commit_message='Phase 1 Boost - Final Stable Version', commit_description='', oid='1bb7b93eb35950babe031720987a13517b5c5572', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Fatoumataa/mt5-bambara-resumer-boost1', endpoint='https://huggingface.co', repo_type='model', repo_id='Fatoumataa/mt5-bambara-resumer-boost1'), pr_revision=None, pr_num=None)

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import pandas as pd

# 1. CONFIGURATION
model_id = "Fatoumataa/mt5-bambara-resumer-boost1"
dataset_path = "/content/dataset_phase1_final_READY.jsonl"
device = "cuda" if torch.cuda.is_available() else "cpu"

# 2. CHARGEMENT DU MODÈLE ET TOKENIZER
print(f"⏳ Chargement du modèle depuis {model_id}...")
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

# 3. CHARGEMENT DE QUELQUES EXEMPLES
df = pd.read_json(dataset_path, lines=True)
# On prend 8 exemples au hasard dans le dataset pour tester la généralisation
samples = df.sample(8, random_state=42)

def generate_summary(text):
    inputs = tokenizer(
        "summarize: " + text,
        return_tensors="pt",
        max_length=512,
        truncation=True
    ).to(device)

    outputs = model.generate(
        inputs["input_ids"],
        max_length=128,
        num_beams=4,
        repetition_penalty=2.5, # Pour éviter les répétitions type "TinyLlama"
        early_stopping=True
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# 4. EXÉCUTION DES TESTS
print("\n" + "="*50)
print("🚀 TEST DU MODÈLE BOOSTÉ (8 EXEMPLES)")
print("="*50 + "\n")

for i, row in samples.iterrows():
    input_text = row['input_text']
    target_text = row['target_text']
    generated_text = generate_summary(input_text)

    print(f"📝 EXEMPLE {i}")
    print(f"🔹 ENTRÉE : {input_text[:150]}...") # Tronqué pour l'affichage
    print(f"✅ ATTENDU (HUMAIN) : {target_text}")
    print(f"🤖 GÉNÉRÉ (IA) : {generated_text}")
    print("-" * 30)

⏳ Chargement du modèle depuis Fatoumataa/mt5-bambara-resumer-boost1...


config.json:   0%|          | 0.00/803 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/16.0M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.23G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/868 [00:00<?, ?B/s]


🚀 TEST DU MODÈLE BOOSTÉ (8 EXEMPLES)

📝 EXEMPLE 16537
🔹 ENTRÉE : summarize_fr: question: quelles sont les caractéristiques des produits de l'agriculture à forte valeur ajoutée et quel est le risque économique majeur...
✅ ATTENDU (HUMAIN) : les produits fva se vendent cher grâce à leur qualité , leur rareté ou leur mode de production . ces productions sont exigeantes en savoir-faire et capital . le risque économique majeur est la **fraude** et la **contrefaçon** des produits, ce qui nécessite une protection rigoureuse des indications géographiques .
🤖 GÉNÉRÉ (IA) : l'agriculture à forte valeur ajoutée est stratégique pour les pays développés car elles permettent de compenser la concurrence des pays à faibles coûts de main d'œuvre et de générer des revenus élevés . cependant, elles sont vulnérables aux **fraude** et à la contrefaçon** .
------------------------------
📝 EXEMPLE 4438
🔹 ENTRÉE : summarize_bm: question: silamɛw ka kalanko ɲɛmɔgɔso ka laɲini tun ye mun ye ani a tun b'a ɲini 

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. SETUP
model_id = "Fatoumataa/mt5-bambara-resumer-boost1"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

def generate_summary(text, task_prefix):
    full_input = task_prefix + text
    inputs = tokenizer(full_input, return_tensors="pt", max_length=512, truncation=True).to(device)
    outputs = model.generate(
        inputs["input_ids"],
        max_length=150,
        num_beams=5,
        repetition_penalty=2.5,
        early_stopping=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# 2. LES 10 EXEMPLES INÉDITS (NON VUS DURANT L'ENTRAÎNEMENT)
tests = [
    # --- BAMBARA ---
    {"prefix": "summarize_bm: ", "text": "question: mun na mɔgɔw ka kan ka ji saniya sanni u k'a min? contexte: ji saniyabali bɛ bana caman lase mɔgɔ ma i n'a fɔ kɔnɔboli. ni mɔgɔ bɛ ji saniya ni fura ye walama k'a gwen, o bɛ kɔrɔni ni fali tɔgɔw bɔ a la ka kɛnɛya lase mɔgɔ ma."},
    {"prefix": "summarize_bm: ", "text": "question: jamana ka sɔrɔ bɛ se ka bonya cogo di ni sɛnɛkɛlaw dɛmɛna? contexte: sɛnɛkɛlaw ye jamana fanga jɔnjɔn ye. ni gɔfɛrɛnaman bɛ masiniw ni bin-fura nɔgɔmanw di sɛnɛkɛlaw ma, o bɛ sɛnɛ kɛ nɔgɔ ye ka sɔrɔ caman lase u ma ka sɔrɔ jɛmɛ bonya."},
    {"prefix": "summarize_bm: ", "text": "question: lakɔli ka nafa ye mun ye denmisɛnw bolo? contexte: lakɔli ye yɔrɔ ye yiriwa bɛ sɔrɔ yɔrɔ min na. kalan fɛ, denmisɛnw bɛ hakili sɔrɔ ka kɛ mɔgɔ dɔnnikɛlaw ye minnu bɛna jamana jɔ sini."},
    {"prefix": "summarize_bm: ", "text": "question: kalanko bɛ kɛ cogo di ni tɛmɛsira tɛ yɔrɔ dɔw la? contexte: dugu dɔw la, tɛmɛsira jugu bɛ to denmisɛnw tɛ se ka lakɔli ta cogo ɲuman na sama fɛ. o bɛ a to u bɛ tila kalan na."},
    {"prefix": "summarize_bm: ", "text": "question: cogo di mɔgɔw bɛ se ka u yɛrɛ tanga nɔgɔ ma dugu kɔnɔ? contexte: dugu saniya ye mɔgɔ bɛɛ kunkan baara ye. ni mɔgɔw bɛ kɔnɔ bɔ yɔrɔ dɔgɔn, k'a to u tɛ nɔgɔ bila tɛmɛsira na, o bɛ dugu kɛ nɔgɔ ye."},

    # --- FRANÇAIS ---
    {"prefix": "summarize_fr: ", "text": "question: quels sont les dangers de la déforestation pour le climat local? contexte: la coupe abusive des arbres entraîne la disparition de l'ombre et la baisse des précipitations. sans forêts, le sol devient stérile et la chaleur augmente, ce qui nuit gravement à l'agriculture locale."},
    {"prefix": "summarize_fr: ", "text": "question: quel est l'impact de l'éducation des filles sur le développement d'une communauté? contexte: instruire une fille, c'est instruire une nation. les femmes éduquées participent activement à l'économie et veillent mieux à la santé et à l'éducation de leurs propres enfants."},
    {"prefix": "summarize_fr: ", "text": "question: pourquoi la vaccination est-elle essentielle pour la santé publique? contexte: la vaccination permet d'éradiquer des maladies mortelles. elle protège non seulement l'individu vacciné mais empêche aussi la propagation du virus au reste de la population."},
    {"prefix": "summarize_fr: ", "text": "question: comment la technologie peut-elle aider les agriculteurs maliens? contexte: l'usage des téléphones mobiles permet aux paysans de connaître les prix du marché en temps réel et de prévoir les pluies grâce aux applications météo, limitant ainsi les pertes."},
    {"prefix": "summarize_fr: ", "text": "question: quel est le rôle de la culture dans le maintien de la paix sociale? contexte: la culture et les traditions, comme le cousinage à plaisanterie, permettent de résoudre les conflits sans violence et de renforcer les liens entre les différentes ethnies."}
]

# 3. LANCEMENT DES TESTS
print(f"{'TYPE':<15} | {'RÉSUMÉ GÉNÉRÉ'}")
print("-" * 80)
for t in tests:
    res = generate_summary(t["text"], t["prefix"])
    print(f"{t['prefix']:<15} | {res}")

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


TYPE            | RÉSUMÉ GÉNÉRÉ
--------------------------------------------------------------------------------
summarize_bm:   | ji saniyabali bɛ bana caman lase mɔgɔ ma i n'a fɔ kɔnɔboli ye walama k'a gwen, o bɛ kɔrɔni ni fali tɛgɔw bɔ a la ka kɛnɛya lase
summarize_bm:   | jamana ka sɔrɔ bɛ se ka bonya cogo di ni bin-fura nɔgɔmanw di sɛnɛkɛlaw dɛmɛna, o bɛ kɛ sababu ye ka jɛmɛ bonya la
summarize_bm:   | lakɔli ye yiriwa bɛ hakili sɔrɔ ka kɛ mɔgɔ dɔnnikɛlaw ye minnu bɛna jamana jɔ sini
summarize_bm:   | kalanko bɛ to denmisɛnw tɛ se ka lakɔli ta cogo ɲuman na sama fɛ
summarize_bm:   | mɔgɔw bɛ kɔnɔ bɔ yɔrɔ dɔgɔn, k'a to u tɛmɛsira na o bɛ dugu kɛ nɔngɔ ye.
summarize_fr:   | les dangers de la déforestation sont la disparition de l'ombre et la baisse des précipitations, ce qui nuit gravement au climat local.
summarize_fr:   | l'éducation des filles est instruire une nation. les femmes éduquées participent activement à la santé et veillent mieux à le développement d'une communauté, cont

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# 1. SETUP
model_id = "Fatoumataa/mt5-bambara-resumer-boost1"
device = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=False)
model = AutoModelForSeq2SeqLM.from_pretrained(model_id).to(device)

def generate_summary(text, task_prefix):
    full_input = task_prefix + text
    inputs = tokenizer(full_input, return_tensors="pt", max_length=512, truncation=True).to(device)
    outputs = model.generate(
        inputs["input_ids"],
        max_length=150,
        num_beams=5,
        repetition_penalty=2.5,
        early_stopping=True
    )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

# 2. LES 10 EXEMPLES DE HAUT NIVEAU
tests = [
    # --- BAMBARA (Sujets : Histoire, Civisme, Économie) ---
    {"prefix": "summarize_bm: ", "text": "question: mun ye mali mansamara ka duga ye taarikisɛbɛn kɔnɔ? contexte: mali mansamara tun ye jamana fanga bon dɔ ye a kɛtɔ ka sanu ni tɛmɛsira bɛɛ mara. mansaw i n'a fɔ sunjata kɛyita y'a to hɛrɛ ni dɛmɛ don mɔgɔw ni ɲɔgɔn cɛ."},
    {"prefix": "summarize_bm: ", "text": "question: mun na sariya mara wajibiyalen don jamana kɔnɔ? contexte: sariya bɛ mɔgɔw kelen-kelen bɛɛ taga yɔrɔ mɔgɔ tɛ se ka o yɔrɔ tɛmɛ. ni sariya tɛ, jamana bɛ kɛ binkannikɛlaw ni fanga-fɛn ta mɔgɔw kɔnɔ."},
    {"prefix": "summarize_bm: ", "text": "question: cogo di bololabaara bɛ se ka mɔgɔ dɛmɛ ka sɔrɔ sɔrɔ? contexte: bololabaara i n'a fɔ kalasɛbɛnni ni kɔ-minɛ-baara bɛ mɔgɔw dɛmɛ ka fɛn dɔw bɔ u bolo k'u feere walasa ka wari sɔrɔ ka u ka dugu yiriwa."},
    {"prefix": "summarize_bm: ", "text": "question: mun ye bɔnɛ ye ni mɔgɔw tɛ dusu-diya kɛ jamana kɔnɔ? contexte: hɛrɛ ni kulu-kelenya ye sababu ye dugu ka kɛ kelen ye. ni dusu-diya ni teriya bɔra mɔgɔw cɛ, o bɛ kɛ sababu ye ka kɛlɛ ni pɛrɛn don jamana kɔnɔ."},
    {"prefix": "summarize_bm: ", "text": "question: mun na kɔgɔji-da jamana bɛ se ka sɔrɔ sɔrɔ ka tɛmɛ Mali kan? contexte: mali ye jamana ye min datugulen don (enclave). o bɛ a to mɔgɔ bɛ fɛn caman min bɔ jamana kɔnɔ, u bɛ wari caman bɔ tɛmɛsira na sanni u ka se kɔgɔji la."},

    # --- FRANÇAIS (Sujets : Droit, Science, Gestion, Citoyenneté) ---
    {"prefix": "summarize_fr: ", "text": "question: en quoi le vote est-il un devoir citoyen essentiel? contexte: voter permet aux citoyens de choisir leurs dirigeants et d'influencer les décisions politiques. c'est un acte de participation à la vie démocratique qui garantit la légitimité des institutions."},
    {"prefix": "summarize_fr: ", "text": "question: quel est l'effet de l'inflation sur le pouvoir d'achat des ménages? contexte: l'inflation est la hausse générale des prix des biens et services. quand les prix augmentent plus vite que les salaires, les familles peuvent acheter moins de produits, ce qui dégrade leur niveau de vie."},
    {"prefix": "summarize_fr: ", "text": "question: pourquoi la protection des données personnelles est-elle cruciale sur internet? contexte: avec le numérique, les informations privées peuvent être volées ou utilisées à des fins commerciales sans accord. protéger ses données évite l'usurpation d'identité et préserve la vie privée."},
    {"prefix": "summarize_fr: ", "text": "question: quel est le rôle des enzymes dans la digestion humaine? contexte: les enzymes sont des protéines qui accélèrent les réactions chimiques. dans l'estomac, elles décomposent les aliments en nutriments plus petits pour qu'ils soient absorbés par le sang."},
    {"prefix": "summarize_fr: ", "text": "question: comment la gestion des déchets contribue-t-elle à la santé urbaine? contexte: ramasser et recycler les ordures permet d'éviter la prolifération des moustiques et des maladies comme le paludisme. une ville propre réduit les dépenses de santé des habitants."}
]

# 3. LANCEMENT DES TESTS
print(f"{'TYPE':<15} | {'RÉSUMÉ GÉNÉRÉ'}")
print("-" * 100)
for t in tests:
    res = generate_summary(t["text"], t["prefix"])
    print(f"{t['prefix']:<15} | {res}")

Loading weights:   0%|          | 0/192 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


TYPE            | RÉSUMÉ GÉNÉRÉ
----------------------------------------------------------------------------------------------------
summarize_bm:   | mali mansamara tun ye jamana fanga bon dɔ ye a kɛtɔ ka sanu ni tɛmɛsira bɛɛ mara walasa ka don mɔgɔw ni ɲɔgɔn cɛ
summarize_bm:   | sariya mara wajibiyalen don jamana kɔnɔ ka binkannikɛlaw ni fanga-fɛn ta mɔgɔw kɔn
summarize_bm:   | bololabaara i n'a fɔ kalasɛbɛnni ni kɔ-minɛ-baara bɛ mɔgɔw dɛmɛ ka fɛn dɔw bɔ u bolo k'u feere walasa ka dugu yiriwa
summarize_bm:   | o ye bɔnɛ ye ni mɔgɔw tɛ dusu-diya kɛ jamana kɔnɔ o bɛ kɛ sababu ye ka kɛlɛ ni pɛrɛn don jamana
summarize_bm:   | kɔgɔji-da jamana bɛ se ka sɔrɔ ka tɛmɛ mali kan o bɛ fɛn caman min bɔ jamana kɔnɔ, u bɛ wari caman bɔ mɔgɔ na sanni u ka se kɔw la
summarize_fr:   | le vote permet aux citoyens de choisir leurs dirigeants et d'influencer les décisions politiques. il est un devoir citoyen essentiel qui garantit la légitimité des institutions, garantissant la légitimination des instit